# Global Flux Jacobian Assembler — Verification Notebook (v4)

**Case:** 2D cylinder, Re = 60, Ma = 0.2, 21 228 nodes  
**Case:** OAT15, Re = 3*10^6, Ma = 0.73, AoA = 3.5 deg

**Purpose:** Verify the assembled sparse Jacobian J = dR/dU via a matrix-free J·q product check.

---

## Key Updates in v4

| # | Change | Details |
|---|--------|---------|
| 1 | **Harten entropy fix in `assemble_residual`** | `assemble_residual` now accepts `harten_eps=0.0`. Interior faces pass `harten_eps` to `inviscid_flux_jacobians`; riemann BC passes it to `absolute_normal_jacobian`. Consistent with the Jacobian assembler. |
| 2 | **Harten fix in `fd_jacobian_col`** | `harten_eps` propagated through `fd_jacobian_col` so FD and analytical Jacobians use the same scheme. |
| 3 | **Bad-node detection (full J)** | Non-finite entries detected across all blocks (diagonal and off-diagonal) via COO format. Bad-node DOFs are zeroed in both LHS and RHS before error reporting. |
| 4 | **Bad-node exclusion in error metrics** | Global, interior, boundary, and per-surface relative errors all exclude bad-node DOFs via `filter_good()`, preventing artificially inflated error norms. |

---

## Key Updates in v3

| # | Change | Details |
|---|--------|---------|
| 1 | **Exact slip-wall flux Jacobian** | `slip_wall_flux_jacobian` uses the analytic chain rule dF/dV\|_W @ dV/dU instead of Roe FD. F_W = [0, P·nx, P·ny, P·nz, 0]ᵀ (u_n = 0). |
| 2 | **Consistent slip flux in `assemble_residual`** | Slip boundary flux now uses the exact pressure at U_i: F_bc = [0, P_i·nx, P_i·ny, P_i·nz, 0]ᵀ — consistent with the Jacobian formulation. |
| 3 | **Cell volume normalisation** | J_phys = diag(1/Vol) @ J_raw converts units from [flux·area] to [1/s]. Missing j-only node volumes are filled by a geometric dual-mesh approximation. |
| 4 | **Area scaling** | All face contributions (interior + boundary) are multiplied by face area before accumulation. |
| 5 | **Relative eps scaling** | FD perturbation h_k = eps × max(|U_k|, eps_abs) per DOF — avoids cancellation in coarse-mesh regions. |

---

## Verification: J·q Product Check

$$
J \, \mathbf{q} \;\approx\; \frac{R(U + h\,\mathbf{q}) - R(U - h\,\mathbf{q})}{2h}
\qquad h = \varepsilon \,\frac{\|U\|}{\|\mathbf{q}\|}
$$

- **q** — random vector of shape (5N,), seed 42
- **LHS** — `J @ q_full` where J is the pre-assembled J_phys (loaded from `.npz`)
- **RHS** — finite difference of `assemble_residual`, then volume-normalised: `D_inv * jv_rhs_raw`
- **Switch** — `use_central = True` (central, O(h²)) or `False` (forward, O(h))
- **eps** — `eps_jv = 1e-8`
- **harten_eps** — must match the value used when assembling J (default 0.0)

> Boundary errors are expected to be larger due to the frozen-ghost approximation at Riemann boundaries.  
> Bad nodes (NaN/inf in J) are excluded from all error norms.

---

## Error Reporting

| Level | Description |
|-------|-------------|
| **Global** | Relative L2 error over all 5N DOFs, bad nodes excluded |
| **Interior nodes** | Excludes boundary and bad nodes |
| **Per surface (flag)** | Broken down by BC type (riemann / slip / noslip), bad nodes excluded |
| **Per node (mesh)** | log₁₀(rel error) mapped onto PyVista mesh; bad nodes highlighted in yellow |

---

## Required Functions

- `euler_flux()`, `inviscid_flux_jacobians()`, `absolute_normal_jacobian()`, `roe_average()`, `_harten()`
- `viscous_flux_jacobians_fd()`, `Fnv_numeric()`, `cons_to_prim()`
- `ghost_state()`, `ghost_state_jacobian_fd()`, `riemann_invariant_bc()`
- `slip_wall_flux_jacobian()`, `dV_dU()`
- `boundary_flux_jacobian_fd()`, `assemble_global_jacobian_fd()`
- `assemble_residual()` ← updated in v4 (harten_eps)
- `extract_cell_volumes()`, `build_boundary_list()`, `boundary_geometry()`, `area_normals()`
- `node_to_dof()`, `filter_good()` ← new in v4

In [2]:
import numpy as np

# pyau3d
from pyau3d.utils import PltFileUtils, GrpFileUtils, UnkFileUtils
from pyau3d.pv.loader.au3d import arrays2vtk

# matplotlib
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import pandas as pd

# scipy
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigs
from scipy.io import mmwrite, mmread
from scipy.spatial import KDTree # for calculating closet point to x,y point
from scipy.sparse import save_npz, load_npz

import time
import pyvista as pv

# %matplotlib widget

In [3]:
# =============================================================================
# GLOBAL FLUX JACOBIAN ASSEMBLER — FINITE DIFFERENCE VERSION (v4)
# Self-contained: all dependencies included.
#
# Updates from v1:
#   (1) viscous_flux_jacobians_fd   — relative eps scaling  (default eps_visc  = 1e-8)
#   (2) ghost_state_jacobian_fd     — relative eps scaling  (default eps_ghost = 1e-6)
#   (3) boundary_flux_jacobian_fd   — exact non-frozen ghost chain rule:
#                                     J_bc = A⁺ + A⁻ @ dUg/dUi  (riemann)
#   (4) assemble_global_jacobian_fd — both eps values exposed in signature
#
# Updates from v2:
#   (5) slip_wall_flux_jacobian     — exact analytic BC Jacobian (replaces FD/Roe):
#                                     J_bc = dF/dV|_W @ dV/dU
#                                     F_W = [0, P·nx, P·ny, P·nz, 0]ᵀ  (u_n = 0)
#   (6) dV_dU                       — primitive/conservative Jacobian (5×5, 3D)
#   (7) assemble_global_jacobian_fd — Area scaling applied to all face contributions
#                                     (fixes factor-of-2 error at corner BC nodes)
#
# Updates from v3:
#   (8) boundary_flux_jacobian_fd   — no-slip wall: non-frozen ghost chain rule:
#                                     J_bc = dFv/dUi + dFv/dUg @ dUg/dUi
#                                     dUg/dUi = diag([1,−1,−1,−1,1])  (exact, no FD)
#   (9) extract_cell_volumes        — two-pass volume extraction: col 5 for i-nodes,
#                                     geometric dual-mesh fallback for j-only nodes
#  (10) assemble_global_jacobian_fd — volume normalisation now internal:
#                                     J_phys = diag(1/Vol) @ J_raw  → [1/s]
#                                     controlled by normalise_volume=True (default)
#
# To tune eps, change only the two arguments in assemble_global_jacobian_fd:
#     eps_visc       : FD step for viscous flux Jacobian          (default 1e-8)
#     eps_ghost      : FD step for Riemann ghost state Jacobian   (default 1e-6)
#     normalise_volume : divide rows by cell volume before return (default True)
# Updates from v4:
#  (11) _harten                     — Harten entropy fix: smooth |λ| near zero
#                                     |λ|_H = (λ²+ε²)/2ε  if |λ| < ε
#  (12) absolute_normal_jacobian    — harten_eps parameter (default 0.0, backward-compatible)
#                                     ε = harten_eps × (|qn| + c)  (local threshold)
#  (13) inviscid_flux_jacobians     — harten_eps forwarded to absolute_normal_jacobian
#  (14) boundary_flux_jacobian_fd   — harten_eps forwarded for riemann BC
#  (15) assemble_global_jacobian_fd — harten_eps exposed in signature (default 0.0)
#
#     harten_eps = 0.0   : disabled (default — no effect on subsonic flows)
#     harten_eps = 0.1–0.3 : recommended for transonic / shock-dominated flows
# =============================================================================

import numpy as np
from scipy.sparse import lil_matrix
from scipy.sparse import diags


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — Inviscid flux Jacobian building blocks
# ─────────────────────────────────────────────────────────────────────────────

def compute_dFdU(U, normal, gamma=1.4):
    """Inviscid flux Jacobian dF/dU at state U in direction normal."""
    rho  = U[0]
    rho_u, rho_v, rho_w, rho_E = U[1], U[2], U[3], U[4]
    nx, ny, nz = normal[0], normal[1], normal[2]
    u = rho_u / rho;  v = rho_v / rho;  w = rho_w / rho
    E = rho_E / rho
    phi = 0.5*(gamma - 1.0)*(u**2 + v**2 + w**2)
    V   = nx*u + ny*v + nz*w
    a1  = gamma*E - phi
    a2  = gamma - 1.0
    a3  = gamma - 2.0
    A   = np.zeros((5, 5))
    A[0, 1] = nx;   A[0, 2] = ny;   A[0, 3] = nz
    A[1, 0] = nx*phi - u*V
    A[1, 1] = V - a3*nx*u;  A[1, 2] = ny*u - a2*nx*v;  A[1, 3] = nz*u - a2*nx*w;  A[1, 4] = a2*nx
    A[2, 0] = ny*phi - v*V
    A[2, 1] = nx*v - a2*ny*u;  A[2, 2] = V - a3*ny*v;  A[2, 3] = nz*v - a2*ny*w;  A[2, 4] = a2*ny
    A[3, 0] = nz*phi - w*V
    A[3, 1] = nx*w - a2*nz*u;  A[3, 2] = ny*w - a2*nz*v;  A[3, 3] = V - a3*nz*w;  A[3, 4] = a2*nz
    A[4, 0] = V*(phi - a1)
    A[4, 1] = a1*nx - a2*u*V;  A[4, 2] = a1*ny - a2*v*V;  A[4, 3] = a1*nz - a2*w*V;  A[4, 4] = gamma*V
    return A

def _harten(lam, eps):
    """Smooth |lambda| near zero: Harten entropy fix."""
    lam = np.asarray(lam, dtype=float)
    if eps == 0.0:
        return np.abs(lam)
    return np.where(np.abs(lam) >= eps,
                    np.abs(lam),
                    (lam**2 + eps**2) / (2.0 * eps))


def absolute_normal_jacobian(U, n, gamma=1.4, harten_eps=0.0):
    """Absolute-value normal Jacobian |A_n| (eqs. 3.6.16–3.6.26)."""
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float);  n = n / np.linalg.norm(n)
    rho = U[0];  vel = U[1:4] / rho;  E = U[4] / rho
    q2  = np.dot(vel, vel)
    p   = (gamma - 1.0)*rho*(E - 0.5*q2)
    c   = np.sqrt(gamma*p/rho)
    H   = E + p/rho
    qn  = np.dot(vel, n)
    M2  = q2/c**2;  Mn = qn/c;  g1 = gamma - 1.0

    # |qn| contribution
    mid = np.zeros((5, 5))
    mid[0, 0]   =  1.0 - 0.5*g1*M2
    mid[0, 1:4] =  (g1/c**2)*vel;   mid[0, 4] = -(g1/c**2)
    mid[1:4, 0]   = -0.5*g1*M2*vel + qn*n
    mid[1:4, 1:4] =  (g1/c**2)*np.outer(vel, vel) + np.eye(3) - np.outer(n, n)
    mid[1:4, 4]   = -(g1/c**2)*vel
    mid[4, 0]   =  qn**2 - 0.5*q2*(1.0 + 0.5*g1*M2)
    mid[4, 1:4] =  (1.0 + 0.5*g1*M2)*vel - qn*n;  mid[4, 4] = -0.5*g1*M2

    # |qn - c| contribution
    l1 = np.empty(5)
    l1[0] = 0.25*g1*M2 + 0.5*Mn;  l1[1:4] = -(g1/(2*c**2))*vel - n/(2*c);  l1[4] = g1/(2*c**2)
    r1 = np.empty(5)
    r1[0] = 1.0;  r1[1:4] = vel - c*n;  r1[4] = H - qn*c
    A1 = np.outer(r1, l1)

    # |qn + c| contribution
    l3 = np.empty(5)
    l3[0] = 0.25*g1*M2 - 0.5*Mn;  l3[1:4] = -(g1/(2*c**2))*vel + n/(2*c);  l3[4] = g1/(2*c**2)
    r3 = np.empty(5)
    r3[0] = 1.0;  r3[1:4] = vel + c*n;  r3[4] = H + qn*c
    A3 = np.outer(r3, l3)
    delta = harten_eps * (abs(qn) + c) 

    return _harten(qn-c, delta)*A1 + _harten(qn, delta)*mid + _harten(qn+c, delta)*A3


def roe_average(UL, UR, gamma=1.4):
    """Roe-averaged conservative state."""
    UL = np.asarray(UL, dtype=float);  UR = np.asarray(UR, dtype=float)
    rhoL, rhoR = UL[0], UR[0]
    velL = UL[1:4]/rhoL;  velR = UR[1:4]/rhoR
    EL = UL[4]/rhoL;      ER  = UR[4]/rhoR
    pL = (gamma-1.0)*rhoL*(EL - 0.5*np.dot(velL, velL))
    pR = (gamma-1.0)*rhoR*(ER - 0.5*np.dot(velR, velR))
    HL = EL + pL/rhoL;  HR = ER + pR/rhoR
    wL = np.sqrt(rhoL);  wR = np.sqrt(rhoR);  ws = wL + wR
    rho_roe = wL*wR
    vel_roe = (wL*velL + wR*velR)/ws
    H_roe   = (wL*HL   + wR*HR  )/ws
    q2_roe  = np.dot(vel_roe, vel_roe)
    E_roe   = (H_roe + (gamma-1.0)*0.5*q2_roe)/gamma
    U_roe   = np.empty(5)
    U_roe[0] = rho_roe;  U_roe[1:4] = rho_roe*vel_roe;  U_roe[4] = rho_roe*E_roe
    return U_roe


def inviscid_flux_jacobians(U_i, U_j, n, gamma=1.4, harten_eps=0.0):
    U_roe     = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=gamma, harten_eps=harten_eps)
    J_L = 0.5*(compute_dFdU(U_i, n, gamma) + abs_A_roe)
    J_R = 0.5*(compute_dFdU(U_j, n, gamma) - abs_A_roe)
    return J_L, J_R


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — Viscous flux and Jacobian
# ─────────────────────────────────────────────────────────────────────────────

def cons_to_prim(U, gamma, R_gas):
    """Conservative → primitive variables."""
    rho = U[0]
    u, v, w = U[1]/rho, U[2]/rho, U[3]/rho
    E = U[4]/rho
    p = (gamma-1.0)*rho*(E - 0.5*(u*u + v*v + w*w))
    T = p/(rho*R_gas)
    return rho, u, v, w, p, T


def Fnv_numeric(U_i, U_j, n, ds,
                gamma=1.4, R_gas=287.0, Pr=0.72,
                mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """Numeric normal viscous flux F_n^v(U_i, U_j, n, ds)."""
    nx, ny, nz = n
    rho_L, u_L, v_L, w_L, p_L, T_L = cons_to_prim(U_i, gamma, R_gas)
    rho_R, u_R, v_R, w_R, p_R, T_R = cons_to_prim(U_j, gamma, R_gas)
    u_avg = (u_L+u_R)/2;  v_avg = (v_L+v_R)/2
    w_avg = (w_L+w_R)/2;  T_avg = (T_L+T_R)/2
    mu_avg  = mu0*(T0+Suth_C)/(T_avg+Suth_C)*(T_avg/T0)**1.5
    dudn = (u_R-u_L)/ds;  dvdn = (v_R-v_L)/ds
    dwdn = (w_R-w_L)/ds;  dTdn = (T_R-T_L)/ds
    div    = dudn*nx + dvdn*ny + dwdn*nz
    tau_xx = mu_avg*(2*dudn*nx - (2/3)*div)
    tau_yy = mu_avg*(2*dvdn*ny - (2/3)*div)
    tau_zz = mu_avg*(2*dwdn*nz - (2/3)*div)
    tau_xy = mu_avg*(dudn*ny + dvdn*nx)
    tau_xz = mu_avg*(dudn*nz + dwdn*nx)
    tau_yz = mu_avg*(dvdn*nz + dwdn*ny)
    tau_nx = tau_xx*nx + tau_xy*ny + tau_xz*nz
    tau_ny = tau_xy*nx + tau_yy*ny + tau_yz*nz
    tau_nz = tau_xz*nx + tau_yz*ny + tau_zz*nz
    tau_nn = tau_nx*u_avg + tau_ny*v_avg + tau_nz*w_avg
    kappa  = gamma*mu_avg/(Pr*(gamma-1.0))
    q_n    = -kappa*dTdn
    return np.array([0., -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n])


def viscous_flux_jacobians_fd(U_i, U_j, n, ds, eps,
                               gamma=1.4, R_gas=287.0, Pr=0.72,
                               mu0=1.716e-5, T0=273.15, Suth_C=110.4,
                               eps_abs=1e-14):
    """
    Forward FD viscous flux Jacobian with relative eps scaling.
    h_k = eps * max(|U[k]|, eps_abs)
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)
    F_base = Fnv_numeric(U_i, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
    J_i_fd = np.zeros((5, 5))
    J_j_fd = np.zeros((5, 5))
    for k in range(5):
        h_i = eps * max(abs(U_i[k]), eps_abs)
        h_j = eps * max(abs(U_j[k]), eps_abs)
        U_i_fwd = U_i.copy();  U_i_fwd[k] += h_i
        J_i_fd[:, k] = (Fnv_numeric(U_i_fwd, U_j, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
                         - F_base) / h_i
        U_j_fwd = U_j.copy();  U_j_fwd[k] += h_j
        J_j_fd[:, k] = (Fnv_numeric(U_i, U_j_fwd, n, ds, gamma, R_gas, Pr, mu0, T0, Suth_C)
                         - F_base) / h_j
    return J_i_fd, J_j_fd


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — Ghost state and its Jacobian
# ─────────────────────────────────────────────────────────────────────────────

def riemann_invariant_bc(U_int, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas=287.0):
    """Ghost cell state via Riemann invariant BC (4-regime selection)."""
    U_int = np.asarray(U_int, dtype=float)
    n     = np.asarray(n,     dtype=float);  n = n / np.linalg.norm(n)
    cv    = R_gas/(gamma - 1.0)
    rho_i = U_int[0];  vel_i = U_int[1:4]/rho_i;  E_i = U_int[4]/rho_i
    Vn_i  = np.dot(vel_i, n);  Vt_i = vel_i - Vn_i*n
    p_i   = (gamma-1.0)*rho_i*(E_i - 0.5*np.dot(vel_i, vel_i))
    T_i   = p_i/(rho_i*R_gas);  c_i = np.sqrt(gamma*R_gas*T_i)
    rho_b = P_b/(R_gas*T_b);  vel_b = np.array([u_b, v_b, w_b])
    Vn_b  = np.dot(vel_b, n);  Vt_b = vel_b - Vn_b*n
    c_b   = np.sqrt(gamma*R_gas*T_b)
    fac   = 2.0/(gamma - 1.0)
    Rp_int = Vn_i + fac*c_i;  Rm_int = Vn_i - fac*c_i
    Rp_b   = Vn_b + fac*c_b;  Rm_b   = Vn_b - fac*c_b
    Mn_i   = Vn_i/c_i
    if Mn_i <= -1.0:
        Vn_wall = Vn_b;  c_wall = c_b;  Vt_wall = Vt_b
        s_wall  = P_b/(rho_b**gamma)
    elif Mn_i >= 1.0:
        Vn_wall = Vn_i;  c_wall = c_i;  Vt_wall = Vt_i
        s_wall  = p_i/(rho_i**gamma)
    elif Vn_i < 0.0:
        Vn_wall = 0.5*(Rp_int + Rm_b);  c_wall = 0.25*(gamma-1.0)*(Rp_int - Rm_b)
        Vt_wall = Vt_b;  s_wall = P_b/(rho_b**gamma)
    else:
        Vn_wall = 0.5*(Rp_int + Rm_b);  c_wall = 0.25*(gamma-1.0)*(Rp_int - Rm_b)
        Vt_wall = Vt_i;  s_wall = p_i/(rho_i**gamma)
    if c_wall <= 0.0:
        raise ValueError(f"Non-physical c_wall={c_wall:.4f}")
    rho_wall = (c_wall**2/(gamma*s_wall))**(1.0/(gamma-1.0))
    p_wall   = s_wall*rho_wall**gamma
    T_wall   = p_wall/(rho_wall*R_gas)
    vel_wall = Vn_wall*n + Vt_wall
    E_wall   = cv*T_wall + 0.5*np.dot(vel_wall, vel_wall)
    U_wall_c = np.array([rho_wall, rho_wall*vel_wall[0],
                          rho_wall*vel_wall[1], rho_wall*vel_wall[2],
                          rho_wall*E_wall])
    return 2.0*U_wall_c - U_int


def ghost_state(U, bc_type, n, gamma=1.4, R_gas=287.0,
                u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """Ghost cell state for slip / noslip / riemann BCs."""
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float);  n = n / np.linalg.norm(n)
    rho = U[0];  vel = U[1:4]/rho;  rho_E = U[4]
    if bc_type == 'slip':
        vn = np.dot(vel, n)
        return np.array([rho, *(rho*(vel - 2.0*vn*n)), rho_E])
    elif bc_type == 'noslip':
        return np.array([rho, *(-rho*vel), rho_E])
    elif bc_type == 'riemann':
        return riemann_invariant_bc(U, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'.")


def ghost_state_jacobian_fd(U_i, bc_type, n,
                             gamma=1.4, R_gas=287.0,
                             u_b=0.0, v_b=0.0, w_b=0.0,
                             T_b=288.15, P_b=101325.0,
                             eps=1e-6,
                             eps_abs=1e-14):
    """
    dU_ghost/dU_i (5×5) via forward FD with relative eps scaling.
    h_k = eps * max(|U_i[k]|, eps_abs)
    Differentiates the actual ghost_state() — consistent with assemble_residual.
    """
    U_i    = np.asarray(U_i, dtype=float)
    n      = np.asarray(n,   dtype=float);  n = n / np.linalg.norm(n)
    Ug_base = ghost_state(U_i, bc_type, n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
    dUg     = np.zeros((5, 5))
    for k in range(5):
        h  = eps * max(abs(U_i[k]), eps_abs)
        ej = np.zeros(5);  ej[k] = h
        Ug_fwd    = ghost_state(U_i + ej, bc_type, n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
        dUg[:, k] = (Ug_fwd - Ug_base) / h
    return dUg      # (5,5)


# EXACT FORMULATION OF SLIP WALL FLUX JACOBIAN

def dV_dU(rho, u, v, w, gamma):
    """
    Jacobian of primitive V=[ρ,u,v,w,p] w.r.t. conservative U=[ρ,ρu,ρv,ρw,ρE]
    5×5 matrix (3-D extension of Eq. 4.32)
    """
    g1  = gamma - 1.0
    q2  = u**2 + v**2 + w**2
    return np.array([
        [ 1.0,       0.0,    0.0,    0.0,   0.0 ],   # ∂ρ/∂U
        [-u/rho,  1.0/rho,   0.0,    0.0,   0.0 ],   # ∂u/∂U
        [-v/rho,    0.0,  1.0/rho,   0.0,   0.0 ],   # ∂v/∂U
        [-w/rho,    0.0,    0.0,  1.0/rho,  0.0 ],   # ∂w/∂U
        [ 0.5*q2*g1, -u*g1, -v*g1, -w*g1,  g1  ],   # ∂p/∂U
    ])

def slip_wall_flux_jacobian(U_i, n, gamma=1.4):
    """
    Exact slip-wall boundary flux Jacobian  dF_bc/dU_i  (5×5).

    Slip-wall flux: F = [0, P*nx, P*ny, P*nz, 0]^T  (u_n = 0 → no convection)
    So  dF/dV|_W  is nonzero only in the pressure column (index 4):
        row 1 → nx,  row 2 → ny,  row 3 → nz,  all others 0

    Chain rule:  dF/dU = dF/dV|_W  @  dV/dU

    Parameters
    ----------
    U_i : (5,)  conservative state [ρ, ρu, ρv, ρw, ρE]
    n   : (3,)  outward unit normal  [nx, ny, nz]
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float)
    n   = n / np.linalg.norm(n)           # ensure unit normal
    nx, ny, nz = n

    rho = U_i[0]
    u, v, w = U_i[1]/rho, U_i[2]/rho, U_i[3]/rho

    # dF/dV|_W — pressure-column-only, 5×5
    dFdV_W          = np.zeros((5, 5))
    dFdV_W[1, 4]    = nx
    dFdV_W[2, 4]    = ny
    dFdV_W[3, 4]    = nz
    # rows 0 and 4 stay zero (mass and energy fluxes vanish at slip wall)

    return dFdV_W @ dV_dU(rho, u, v, w, gamma)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — Boundary flux Jacobian (non-frozen ghost)
# ─────────────────────────────────────────────────────────────────────────────

def boundary_flux_jacobian_fd(U_i, bc_type, n, ds,
                               eps_visc, eps_ghost,
                               gamma=1.4, R_gas=287.0,
                               Pr=0.72, mu0=1.716e-5,
                               T0=273.15, Suth_C=110.4,
                               u_b=0.0, v_b=0.0, w_b=0.0,
                               T_b=300.0, P_b=101325.0,
                               harten_eps=0.0): 
    """
    Boundary flux Jacobian dF_bc/dU_i  (5×5).

    slip    — exact analytic:  J_bc = dF/dV|_W @ dV/dU
    riemann — non-frozen ghost chain rule:  J_bc = A⁺ + A⁻ @ dUg/dUi
    noslip  — viscous FD Jacobian
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float);  n = n / np.linalg.norm(n)

    # ── slip: exact analytic — no ghost state needed ──────────────────────────
    if bc_type == 'slip':
        return slip_wall_flux_jacobian(U_i, n, gamma)

    # ── riemann: non-frozen ghost chain rule ──────────────────────────────────
    elif bc_type == 'riemann':
        U_g = ghost_state(U_i, 'riemann', n, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
        J_L, J_R = inviscid_flux_jacobians(U_i, U_g, n, gamma, harten_eps=harten_eps) 
        dUg_dUi  = ghost_state_jacobian_fd(
            U_i, 'riemann', n,
            gamma=gamma, R_gas=R_gas,
            u_b=u_b, v_b=v_b, w_b=w_b,
            T_b=T_b, P_b=P_b,
            eps=eps_ghost)
        return J_L + J_R @ dUg_dUi

    # ── noslip: viscous FD ────────────────────────────────────────────────────
    elif bc_type == 'noslip':
        U_g = ghost_state(U_i, 'noslip', n)
        J_i_visc, J_g_visc = viscous_flux_jacobians_fd(
            U_i, U_g, n, max(ds, 1e-14),
            eps=eps_visc, gamma=gamma, R_gas=R_gas,
                Pr=Pr, mu0=mu0, T0=T0, Suth_C=Suth_C)
        
        # exact dUg/dUi for noslip — no FD needed
        dUg_dUi = np.diag([1.0, -1.0, -1.0, -1.0, 1.0])

        return J_i_visc + J_g_visc @ dUg_dUi   # full non-frozen

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'.")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — Global assembler  ← main entry point
#
# Tune eps here:
#   eps_visc  = 1e-8   (viscous flux FD step)
#   eps_ghost = 1e-6   (ghost state FD step)
# ─────────────────────────────────────────────────────────────────────────────

def extract_cell_volumes(fortfile, coord, N):
    """
    Pass 1 : fill from col 5 wherever node appears as i
    Pass 2 : for nodes that never appear as i, estimate from
             face area × ds contributions (geometric dual-mesh approx)
    """
    cell_vol = np.zeros(N)

    # ── Pass 1: direct from fortfile col 5 ───────────────────────────────────
    for row in fortfile:
        i = int(row[0]) - 1
        cell_vol[i] = row[5]            # Vol_i always reliable

    # ── Pass 2: geometric fallback for j-only nodes ───────────────────────────
    # Each face contributes 0.5 * Area * ds to both its nodes
    # Only fills nodes that still have vol == 0
    geo_vol = np.zeros(N)
    for row in fortfile:
        i    = int(row[0]) - 1
        j    = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area
        ds   = max(abs(np.dot(coord[j] - coord[i], n_ij)), 1e-14)
        slab = 0.5 * Area * ds
        geo_vol[i] += slab
        geo_vol[j] += slab

    # Apply geometric estimate only where col 5 gave nothing
    missing_mask         = (cell_vol == 0)
    cell_vol[missing_mask] = geo_vol[missing_mask]

    # Sanity check
    still_missing = np.where(cell_vol == 0)[0]
    if len(still_missing):
        raise ValueError(f"Volume still zero for nodes: {still_missing}")

    n_filled   = missing_mask.sum()
    n_from_col = N - n_filled
    print(f"Volumes from col 5 (directly extracted from node i ) : {n_from_col}")
    print(f"Volumes from geometry (by geometric approxmiation for missing j-nodes): {n_filled}")

    return cell_vol

def assemble_global_jacobian_fd(fortfile, U_list, coord, boundary_list,
                                 gamma=1.4, R_gas=287.0,
                                 viscous=False,
                                 Pr=0.72, mu0=1.716e-5,
                                 T0=273.15, Suth_C=110.4,
                                 eps_visc=1e-8,
                                 eps_ghost=1e-6,
                                 include_bc=True,
                                 normalise_volume=True,
                                 harten_eps=0.0):   # ← new flag
    
    N       = U_list.shape[0]
    J       = lil_matrix((5*N, 5*N))
    visc_kw = dict(gamma=gamma, R_gas=R_gas, Pr=Pr, mu0=mu0, T0=T0, Suth_C=Suth_C)

    # ── Interior faces ────────────────────────────────────────────────────────
    for row in fortfile:
        i    = int(row[0]) - 1;  j = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area

        J_L, J_R = inviscid_flux_jacobians(U_list[i], U_list[j], n_ij, gamma, harten_eps=harten_eps)

        if viscous:
            ds      = max(abs(float(np.dot(coord[j] - coord[i], n_ij))), 1e-14)
            Jv_L, Jv_R = viscous_flux_jacobians_fd(
                U_list[i], U_list[j], n_ij, ds,
                eps=eps_visc, **visc_kw)
            J_L = J_L + Jv_L;  J_R = J_R + Jv_R

        ri = slice(5*i, 5*i+5);  ci = slice(5*i, 5*i+5)
        rj = slice(5*j, 5*j+5);  cj = slice(5*j, 5*j+5)
        
        J[ri, ci] += J_L * Area
        J[ri, cj] += J_R * Area
        J[rj, ci] -= J_L * Area
        J[rj, cj] -= J_R * Area

    if include_bc:
        # ── Boundary faces ────────────────────────────────────────────────────────
        bc_map = {0: 'riemann', 1: 'slip', 2: 'noslip'}

        for row in boundary_list:
            i       = int(row[0]) - 1
            A_bc = np.asarray(row[1:4], dtype=float)
            Area = np.linalg.norm(A_bc)
            ds   = float(row[4])
            if Area < 1e-14:
                continue
            n_bc = A_bc / Area
            bc_type = bc_map[int(row[5])]
            u_b, v_b, w_b = float(row[6]), float(row[7]), float(row[8])
            T_b, P_b      = float(row[9]),  float(row[10])

            J_bc = boundary_flux_jacobian_fd(
                U_list[i], bc_type, n_bc, ds,
                eps_visc=eps_visc,
                eps_ghost=eps_ghost,
                u_b=u_b, v_b=v_b, w_b=w_b,
                T_b=T_b, P_b=P_b,
                **visc_kw, 
                harten_eps=harten_eps)

            ri = slice(5*i, 5*i+5)
            J[ri, ri] += J_bc * Area

        # ── Volume normalisation ──────────────────────────────────────────────────
    if normalise_volume:
        cell_vol = extract_cell_volumes(fortfile, coord, N)
        D_inv    = np.repeat(1.0 / cell_vol, 5)
        J        = diags(D_inv) @ J     # J_phys in [1/s]

    return J.tocsr()


# ─────────────────────────────────────────────────────────────────────────────
# EXAMPLE CALL
# ─────────────────────────────────────────────────────────────────────────────
# J = assemble_global_jacobian_fd(
#         fortfile, U_list, coord, boundary_list,
#         viscous=True,
#         eps_visc=1e-8,     # increase to 1e-7 if noslip errors are large
#         eps_ghost=1e-6)    # increase to 1e-5 if riemann errors are large

In [4]:
# build_boundary_list function

# Area normal function - helper function

def area_normals(coord, ifac3=None, ifac4=None):
    """
    Compute area-weighted normal vectors (Ax, Ay, Az) at each nodes,
    accumulated from triangle and/or quadrilateral faces. No PolyData is
    built; only the (N, 3) array is returned.
 
    For each face, the area-normal vector is:
        triangle : 0.5 * cross(v1 - v0, v2 - v0)
        quad     : sum of the two triangle area-normals from splitting
                   the quad (0,1,2) + (0,2,3)
    Each face's area-normal is split equally among its vertices and summed.

    That is how it accounted from the node-centered formulation
 
    Args:
        coord : (N, 3) array of mesh point coordinates
        ifac3 : (M3, 3) array of triangle connectivity, zero-based (or None)
        ifac4 : (M4, 4) array of quad connectivity, zero-based (or None)
 
    Returns:
        anor : (N, 3) array of area-weighted normals (Ax, Ay, Az) per point
    """
    coord = np.asarray(coord, dtype=float)
    anor = np.zeros_like(coord)
 
    if ifac3 is not None and len(ifac3) > 0:
        ifac3 = np.asarray(ifac3)
        v0, v1, v2 = coord[ifac3[:, 0]], coord[ifac3[:, 1]], coord[ifac3[:, 2]]
        face_anor = 0.5 * np.cross(v1 - v0, v2 - v0)   # (M3, 3)
        contrib = -face_anor / 3.0                      # sign matches ref code
        for k in range(3):
            np.add.at(anor, ifac3[:, k], contrib)
 
    if ifac4 is not None and len(ifac4) > 0:
        ifac4 = np.asarray(ifac4)
        v0, v1, v2, v3 = (coord[ifac4[:, 0]], coord[ifac4[:, 1]],
                          coord[ifac4[:, 2]], coord[ifac4[:, 3]])
        n1 = 0.5 * np.cross(v1 - v0, v2 - v0)
        n2 = 0.5 * np.cross(v2 - v0, v3 - v0)
        face_anor = n1 + n2                              # (M4, 3)
        contrib = -face_anor / 4.0
        for k in range(4):
            np.add.at(anor, ifac4[:, k], contrib)
 
    return anor


def boundary_geometry(pltfile, coord, fortfile, flag):
    """
    Wall-normal distance for each boundary node on a given surface.

    ds_i = mean over interior neighbours j of | n_hat_i . (x_j - x_i) |

    Returns
    -------
    (Nb, 5) array: [node_id (0-based), Abx, Aby, Abz, ds]

    """
    # use extract_surface_real to ifac4 as well 
    surface_nodes, tri_connect, quad_connect = pltfile.extract_surface_real(flag = flag)

    # get surface coordinates (0 based indexing)
    surface_coord = coord[surface_nodes]

    surface_area_normals = area_normals(surface_coord, ifac3=tri_connect, ifac4 = quad_connect)

    n_hat = surface_area_normals / np.linalg.norm(surface_area_normals, axis=1, keepdims=True)

    # global node id -> row in surface_nodes / n_hat
    local_index = -np.ones(coord.shape[0], dtype=int)
    local_index[surface_nodes] = np.arange(len(surface_nodes))

    surface_mask = np.zeros(coord.shape[0], dtype=bool)
    surface_mask[surface_nodes] = True

    ds_lists = {}   # boundary node id -> list of projected distances to its interior neighbours

    for row in fortfile:
        a = int(row[0]) - 1   # 0-based
        b = int(row[1]) - 1

        a_surf = surface_mask[a]
        b_surf = surface_mask[b]

        if a_surf and not b_surf:
            node, neigh = a, b
        elif b_surf and not a_surf:
            node, neigh = b, a
        else:
            continue   # both on surface (tangential edge) or both interior -> not what we want

        normal = n_hat[local_index[node]]
        vec = coord[neigh] - coord[node]
        ds_proj = abs(np.dot(vec, normal))

        ds_lists.setdefault(node, []).append(ds_proj)

    node_ids = np.array(sorted(ds_lists.keys()))
    ds = np.array([np.mean(ds_lists[n]) for n in node_ids])

    return np.column_stack([node_ids, surface_area_normals, ds])


def build_boundary_list(pltfile, coord, fortfile, flags_bc_map):
    """
    boundary_list : (B, 12) ndarray
        col 0    : node i (1-based)
        col 1-3  : [Ax, Ay, Az]  outward area-weighted normal
        col 4    : ds
        col 5    : bc_type   0=riemann | 1=slip | 2=noslip
        col 6-8  : u_b, v_b, w_b
        col 9    : T_b
        col 10   : P_b
        col 11   : flag (surface id, for filtering/debugging)
    """
    rows = []
    for flag, bc in flags_bc_map.items():
        geom = boundary_geometry(pltfile, coord, fortfile, flag)
        n = geom.shape[0]
        bc_cols = np.tile([
            float(bc['bc_type']), float(bc['u_b']), float(bc['v_b']),
            float(bc['w_b']), float(bc['T_b']), float(bc['P_b']),
        ], (n, 1))
        node_1based = geom[:, [0]] + 1.0
        flag_col = np.full((n, 1), flag, dtype=float)
        rows.append(np.hstack([node_1based, geom[:, 1:5], bc_cols, flag_col]))
        
    return np.vstack(rows)

In [5]:
# =============================================================================
# EULER FLUX  (physical flux in direction n)
# ============================================================================

def euler_flux(U, n, gamma=1.4):
    """
    Computes the Euler flux projected onto normal n. (Euler normal flux)
    U  = [rho, rhou, rhov, rhow, rhoE]
    n  = [nx, ny, nz]  (need not be unit)
    F  = [rho*un, rhou*un + P*nx, rhov*un + P*ny, rhow*un + P*nz, (rhoE+P)*un]
    un = u*nx + v*ny + w*nz
    """
    rho  = U[0]
    rhou = U[1]; rhov = U[2]; rhow = U[3]; rhoE = U[4]

    u = rhou / rho
    v = rhov / rho
    w = rhow / rho

    nx, ny, nz = n[0], n[1], n[2]
    un = u * nx + v * ny + w * nz                        

    P = (gamma - 1.0) * (rhoE - 0.5 * rho * (u**2 + v**2 + w**2))  

    F = np.zeros(5)                                      
    F[0] = rho  * un
    F[1] = rhou * un + P * nx
    F[2] = rhov * un + P * ny
    F[3] = rhow * un + P * nz
    F[4] = (rhoE + P) * un

    return F

# =============================================================================
# RESIDUAL ASSEMBLER
# =============================================================================

def assemble_residual(
        fortfile, U_list, coord, boundary_list,
        gamma=1.4, R_gas=287.0,
        viscous=False,
        Pr=0.72, mu0=1.716e-5, T0=273.15, Suth_C=110.4,
        harten_eps=0.0):                                   # ← add
    """
    Assemble the residual vector R(Q) that corresponds to assemble_global_jacobian.

    Uses the identical face loop structure and boundary_list format as
    assemble_global_jacobian so that the FD Jacobian (dR/dQ computed by
    perturbing Q) matches the analytical Jacobian column-by-column.

    Interior face  (i→j):
        F_Roe = euler_flux(U_i, n) + A⁻ @ (U_j − U_i)   ← Roe flux
        R[i]  += F_Roe * Area
        R[j]  -= F_Roe * Area

    Boundary face on cell i:
        F_bc  = euler_flux(U_i, n) + A⁻_bc @ (U_ghost − U_i)
        R[i]  += F_bc * Area

    Parameters  (identical to assemble_global_jacobian)
    ----------
    fortfile      : ndarray (M, 5) or (M, 8)
    U_list        : ndarray (N, 5)
    coord         : ndarray (N, 3)
    boundary_list : ndarray (B, 12)

    harten_eps    : float, epsilon value harten's fix for roe-flux

    Returns
    -------
    R : ndarray (5N,)   flattened residual vector
    """
    bc_str = {0: 'riemann', 1: 'slip', 2: 'noslip'}
    N      = U_list.shape[0]
    R      = np.zeros((N, 5))

    visc_kw = dict(gamma=gamma, R_gas=R_gas, Pr=Pr,
                   mu0=mu0, T0=T0, Suth_C=Suth_C)

    # ── Interior faces ────────────────────────────────────────────────────────
    for row in fortfile:
        i    = int(row[0]) - 1
        j    = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area

        # Roe flux:  F_L + A⁻ @ dU   (reuses existing Jacobian function)
        F_L       = euler_flux(U_list[i], n_ij, gamma)
        _, dFj = inviscid_flux_jacobians(U_list[i], U_list[j], n_ij, gamma,
                                         harten_eps=harten_eps)   # ← add
        F_face    = F_L + dFj @ (U_list[j] - U_list[i])

        # Viscous contribution
        if viscous:

            ds = abs(float(np.dot(coord[j] - coord[i], n_ij)))
            ds = max(ds, 1e-14)
            F_face = F_face + Fnv_numeric(U_list[i], U_list[j], n_ij, ds, **visc_kw)

        R[i] += F_face * Area
        R[j] -= F_face * Area

    # ── Boundary faces ────────────────────────────────────────────────────────
    for row in boundary_list:
        i    = int(row[0]) - 1
        A_bc = np.asarray(row[1:4], dtype=float)
        Area = np.linalg.norm(A_bc)
        ds   = float(row[4])
        if Area < 1e-14:
            continue
        n_bc = A_bc / Area
        bc   = bc_str[int(row[5])]
        u_b, v_b, w_b, T_b, P_b = (float(row[6]), float(row[7]),
                                        float(row[8]), float(row[9]), float(row[10]))

        # Ghost state (same logic as boundary_flux_jacobian)
        if bc == 'riemann':
            U_g = ghost_state(U_list[i], 'riemann', n_bc, gamma, R_gas,
                              u_b, v_b, w_b, T_b, P_b)
        else:
            U_g = ghost_state(U_list[i], bc, n_bc)

        # Boundary flux
        if bc == 'slip': # slip-wall exact pressure flux by [0, P_i·nx, P_i·ny, P_i·nz, 0]
            rho_i = U_list[i][0]
            u_i, v_i, w_i = U_list[i][1]/rho_i, U_list[i][2]/rho_i, U_list[i][3]/rho_i
            P_i = (gamma-1.0) * (U_list[i][4] - 0.5*rho_i*(u_i**2 + v_i**2 + w_i**2))
            F_bc = np.array([0.0, P_i*n_bc[0], P_i*n_bc[1], P_i*n_bc[2], 0.0])
    
        elif bc == 'riemann':
            F_L       = euler_flux(U_list[i], n_bc, gamma)
            F_g       = euler_flux(U_g, n_bc, gamma)
            U_roe     = roe_average(U_list[i], U_g, gamma)
            abs_A_roe = absolute_normal_jacobian(U_roe, n_bc, gamma=gamma,
                                                 harten_eps=harten_eps)   # ← add
            F_bc      = 0.5*(F_L + F_g) + 0.5 * abs_A_roe @ (U_list[i] - U_g)

        elif bc == 'noslip':
            ds = max(ds, 1e-14)
            F_bc = Fnv_numeric(U_list[i], U_g, n_bc, ds, **visc_kw)

        R[i] += F_bc * Area

    return R.flatten()    # (5N,)

def fd_jacobian_col(j, Q_flat, fortfile, coord, boundary_list,
                    eps=1e-6, eps_abs=1e-14, gamma=1.4, R_gas=287.0,
                    harten_eps=0.0):                                  # ← add
    h     = eps * max(abs(Q_flat[j]), eps_abs)
    ej    = np.zeros_like(Q_flat)
    ej[j] = h
    R_fwd = assemble_residual(fortfile, (Q_flat+ej).reshape(-1,5),
                               coord, boundary_list, gamma=gamma, R_gas=R_gas,
                               harten_eps=harten_eps)                 # ← add
    R_bwd = assemble_residual(fortfile, (Q_flat-ej).reshape(-1,5),
                               coord, boundary_list, gamma=gamma, R_gas=R_gas,
                               harten_eps=harten_eps)                 # ← add
    return (R_fwd - R_bwd) / (2*h)

In [6]:
# utility functions

def specific_energy(rho, p, ux, uy, uz, gamma=1.4):
    return p / ((gamma - 1.0) * rho) + 0.5 * (ux**2 + uy**2 + uz**2)

def conservative_variables(rst, GAMMA):
    """Return conservative variables U on the surface."""
    E = specific_energy(rst.rho, rst.p, rst.ux, rst.uy, rst.uz, GAMMA)

    U = np.column_stack((
        rst.rho,
        rst.rho * rst.ux,
        rst.rho * rst.uy,
        rst.rho * rst.uz,
        rst.rho * E
    ))
    return U

In [11]:
# 1. Read neccessary files
Mesh = 21228
Re   = 60
Mach = 0.2
gamma = 1.4
R_gas = 287.0          # confirm units match the solver
mesh_ver = 1
case_name = "OAT"

# dir = f"C:/Users/User/Git/flux_jacobian/cases/v{mesh_ver}_mesh/cylinder_{Mesh}_Re{Re}_M{Mach}"
# dir = f"/home/ahf25/CFD_2d_cylinder_all/Steady/Ma{Mach}/v{mesh_ver}_mesh/2d_cylinder_{Mesh}_Re{Re}" 
# dir = f"/home/ahf25/OAT15/OAT15_M0.73_A35" 

dir = "C:/Users/User/Git/flux_jacobian/cases/OAT15/OAT15_M0.73_A35"
pltfile = PltFileUtils(f"{dir}/{case_name}.plt")
rstfile = UnkFileUtils(f"{dir}/{case_name}.unk", extend=False)  # both rst and unk are fine

fortfile = pd.read_csv(f"{dir}/fort.864", sep=r'\s+', header=None).to_numpy()

rstfile._primitive()
U_list = conservative_variables(rstfile, gamma)
coord  = pltfile.coord

# u_in  = 68.0525;  v_in  = 0.0;  w_in  = 0.0
# T_in  = 288.15;   P_in  = 1.32702
# u_out = 68.0525;  v_out = 0.0;  w_out = 0.0
# T_out = 288.15;   P_out = 1.32702

# bc_type   0=riemann | 1=slip | 2=noslip
# flags_bc_map = {
#     1: {'bc_type': 0, 'u_b': u_in,  'v_b': v_in,  'w_b': w_in,  'T_b': T_in,  'P_b': P_in},
#     2: {'bc_type': 0, 'u_b': u_out, 'v_b': v_out, 'w_b': w_out, 'T_b': T_out, 'P_b': P_out},
#     3: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     4: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     5: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     6: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
#     7: {'bc_type': 2, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
# }



# for OAT15 M=0.73, AoA = 18569

u_in  = 252.98;  v_in  = 15.47;  w_in  = 0
T_in  = 300;   P_in  = 18569
u_out = 252.98;  v_out = 15.47;  w_out = 0
T_out = 300;   P_out = 18569

# bc_type   0=riemann | 1=slip | 2=noslip
flags_bc_map = {
    1: {'bc_type': 2, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    2: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    3: {'bc_type': 1, 'u_b': 0.0,   'v_b': 0.0,   'w_b': 0.0,   'T_b': T_in,  'P_b': P_in},
    4: {'bc_type': 0, 'u_b': u_in,  'v_b': v_in,  'w_b': w_in,  'T_b': T_in,  'P_b': P_in},
    5: {'bc_type': 0, 'u_b': u_out, 'v_b': v_out, 'w_b': w_out, 'T_b': T_out, 'P_b': P_out},
}

boundary_list = build_boundary_list(pltfile, coord, fortfile, flags_bc_map)


# # Read the jacobians from ./data
# data_dir = f"/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v4/v{mesh_ver}_mesh"
# data_dir = "C:/Users/User/Git/flux_jacobian/data/flux_jacobian_assembly_v4"
# npz_file = f"jacobian_cylinder_{Mesh}_Re{Re}_M{Mach}_fd.npz"

# # data_dir = f"/home/ahf25/git/flux_jacobian/data/flux_jacobian_assembly_v5"
data_dir = "C:/Users/User/Git/flux_jacobian/data/flux_jacobian_assembly_v5"
npz_file = "jacobian_OAT15_M0.73_A35_fd_noharten.npz"

global_flux_jacobian_fd = load_npz(f"{data_dir}/{npz_file}")

J = global_flux_jacobian_fd


In [12]:
# transform PLT file to VTK 
mesh = arrays2vtk(pltfile)

# extract block in multiblock
domain = pv.wrap(mesh.GetBlock(0))    # "Domain 1"  →  pv.MultiBlock
block  = pv.wrap(domain.GetBlock(0))  # "Volume"    →  pv.UnstructuredGrid

In [13]:
# extract cell volume
from scipy.sparse import diags

def extract_cell_volumes(fortfile, coord, N):
    """
    Pass 1 : fill from col 5 wherever node appears as i
    Pass 2 : for nodes that never appear as i, estimate from
             face area × ds contributions (geometric dual-mesh approx)
    """
    cell_vol = np.zeros(N)

    # ── Pass 1: direct from fortfile col 5 ───────────────────────────────────
    for row in fortfile:
        i = int(row[0]) - 1
        cell_vol[i] = row[5]            # Vol_i always reliable

    # ── Pass 2: geometric fallback for j-only nodes ───────────────────────────
    # Each face contributes 0.5 * Area * ds to both its nodes
    # Only fills nodes that still have vol == 0
    geo_vol = np.zeros(N)
    for row in fortfile:
        i    = int(row[0]) - 1
        j    = int(row[1]) - 1
        A_ij = np.asarray(row[2:5], dtype=float)
        Area = np.linalg.norm(A_ij)
        if Area < 1e-14:
            continue
        n_ij = A_ij / Area
        ds   = max(abs(np.dot(coord[j] - coord[i], n_ij)), 1e-14)
        slab = 0.5 * Area * ds
        geo_vol[i] += slab
        geo_vol[j] += slab

    # Apply geometric estimate only where col 5 gave nothing
    missing_mask         = (cell_vol == 0)
    cell_vol[missing_mask] = geo_vol[missing_mask]

    # Sanity check
    still_missing = np.where(cell_vol == 0)[0]
    if len(still_missing):
        raise ValueError(f"Volume still zero for nodes: {still_missing}")

    n_filled   = missing_mask.sum()
    n_from_col = N - n_filled
    print(f"Volumes from col 5 (directly extracted from node i ) : {n_from_col}")
    print(f"Volumes from geometry (by geometric approxmiation for missing j-nodes): {n_filled}")

    return cell_vol

N = U_list.shape[0]
cell_vol = extract_cell_volumes(fortfile, coord, N)

Volumes from col 5 (directly extracted from node i ) : 530075
Volumes from geometry (by geometric approxmiation for missing j-nodes): 10081


In [14]:
# ── Tag bad nodes from full J (diagonal AND off-diagonal blocks) ──────────────
J_coo          = J.tocoo()
nonfinite_mask = ~np.isfinite(J_coo.data)
bad_dofs       = np.unique(np.concatenate([
                     J_coo.row[nonfinite_mask],
                     J_coo.col[nonfinite_mask]
                 ]))
bad_nodes      = np.unique(bad_dofs // 5)
print(f"Bad nodes (full J): {len(bad_nodes)} / {N}")

# boolean mask over all DOFs — False at bad node DOFs
good_mask = np.ones(5*N, dtype=bool)
for n in bad_nodes:
    good_mask[5*n:5*n+5] = False

# ── Clean J: replace NaN/inf with 0 so sparse matvec doesn't propagate NaN ───
J_clean      = J.copy()
J_clean.data = np.where(np.isfinite(J_clean.data), J_clean.data, 0.0)

Bad nodes (full J): 0 / 540156


In [15]:
# COMPUTE LHS AND RHS OF J*v
# Randomly generate q
np.random.seed(42)
q_full = np.random.randn(5 * N)   # (5*N,) fully random

# define U_flat
U_flat = U_list.flatten()

# ── RHS: finite difference of residual, then volume-normalise ─────────────────
eps_jv = 1e-8
h = eps_jv * np.linalg.norm(U_flat) / (np.linalg.norm(q_full) + 1e-14)

use_central = True  # ← switch here: True = central diff, False = forward diff

harten_eps = 0.0

if use_central:
    R_fwd = assemble_residual(fortfile, (U_flat + h*q_full).reshape(-1,5),
                               coord, boundary_list, gamma=gamma, R_gas=R_gas, harten_eps= harten_eps)
    R_bwd = assemble_residual(fortfile, (U_flat - h*q_full).reshape(-1,5),
                               coord, boundary_list, gamma=gamma, R_gas=R_gas, harten_eps= harten_eps)
    jv_rhs_raw = (R_fwd - R_bwd) / (2*h)        # O(h²)
else:
    R0    = assemble_residual(fortfile, U_list,
                               coord, boundary_list, gamma=gamma, R_gas=R_gas, harten_eps= harten_eps)
    R_fwd = assemble_residual(fortfile, (U_flat + h*q_full).reshape(-1,5),
                               coord, boundary_list, gamma=gamma, R_gas=R_gas, harten_eps= harten_eps)
    jv_rhs_raw = (R_fwd - R0) / h               # O(h)

# apply same 1/Vol normalisation as J_phys
D_inv  = np.repeat(1.0 / cell_vol, 5)           # (5N,)
jv_rhs = D_inv * jv_rhs_raw
jv_rhs[~good_mask] = 0.0          # zero out bad node DOFs in RHS

# ── LHS ───────────────────────────────────────────────────────────────────────
jv_lhs = J_clean @ q_full                             # J_phys @ 
jv_lhs[~good_mask]  = 0.0         # zero out bad node DOFs in LHS

In [16]:
print(f"eps for J*q: {eps_jv}")
diff_str = "central" if use_central else "forward"
print(f"By {diff_str} diff, h={h:.2e}")

# ── Extract boundary and interior node indices ────────────────────────────────
boundary_nodes_1based = boundary_list[:, 0].astype(int)
boundary_node_idx     = np.unique(boundary_nodes_1based - 1)   # 0-based

all_node_idx      = np.arange(N)
interior_node_idx = np.setdiff1d(all_node_idx, boundary_node_idx)

# expand node indices to DOF indices (5 DOFs per node)
def node_to_dof(node_idx):
    node_idx = np.asarray(node_idx)
    if node_idx.size == 0:
        return np.array([], dtype=int)
    return np.concatenate([np.arange(5*n, 5*n+5) for n in node_idx])

interior_dof_idx = node_to_dof(interior_node_idx)
boundary_dof_idx = node_to_dof(boundary_node_idx)

# ── Helper: keep only good (non-bad-node) DOFs from any index set ─────────────
def filter_good(dof_idx):
    """Remove bad-node DOFs from dof_idx using the precomputed good_mask."""
    return dof_idx[good_mask[dof_idx]]

# all-DOF index arrays, filtered
all_dof_idx      = filter_good(np.arange(5*N))
interior_dof_idx = filter_good(node_to_dof(interior_node_idx))
boundary_dof_idx = filter_good(node_to_dof(boundary_node_idx))

# ── Global error ───────────────────────────────────────────────────────────────
diff_str = "central" if use_central else "forward"
rel_err  = (np.linalg.norm(jv_lhs[all_dof_idx] - jv_rhs[all_dof_idx])
            / (np.linalg.norm(jv_rhs[all_dof_idx]) + 1e-14))
print(f"JV relative error ({diff_str} diff, h={h:.2e}): {rel_err:.4e}")
print(f"  ||jv_lhs|| = {np.linalg.norm(jv_lhs[all_dof_idx]):.6e}")
print(f"  ||jv_rhs|| = {np.linalg.norm(jv_rhs[all_dof_idx]):.6e}")
print(f"  Bad nodes excluded: {len(bad_nodes)}")

# ── Interior nodes only ────────────────────────────────────────────────────────
jv_lhs_int  = jv_lhs[interior_dof_idx]
jv_rhs_int  = jv_rhs[interior_dof_idx]
rel_err_int = (np.linalg.norm(jv_lhs_int - jv_rhs_int)
               / (np.linalg.norm(jv_rhs_int) + 1e-14))
print(f"\nInterior nodes ({len(interior_node_idx)}, bad excluded) rel err: {rel_err_int:.4e}")
print(f"  ||jv_lhs|| = {np.linalg.norm(jv_lhs_int):.4e}")
print(f"  ||jv_rhs|| = {np.linalg.norm(jv_rhs_int):.4e}")

# ── All boundary nodes ─────────────────────────────────────────────────────────
jv_lhs_bc  = jv_lhs[boundary_dof_idx]
jv_rhs_bc  = jv_rhs[boundary_dof_idx]
rel_err_bc = (np.linalg.norm(jv_lhs_bc - jv_rhs_bc)
              / (np.linalg.norm(jv_rhs_bc) + 1e-14))
print(f"\nBoundary nodes ({len(boundary_node_idx)}, bad excluded) rel err: {rel_err_bc:.4e}")

# ── Per-surface error ──────────────────────────────────────────────────────────
flags        = boundary_list[:, 11].astype(int)
unique_flags = np.unique(flags)
bc_type_str  = {0: 'riemann', 1: 'slip', 2: 'noslip'}

print(f"\n{'Flag':<6} {'BC type':<10} {'N nodes':>8} "
      f"{'||lhs||':>12} {'||rhs||':>12} {'rel_err':>12}")
print("-" * 64)
for flag in unique_flags:
    mask         = flags == flag
    surf_nodes   = np.unique(boundary_list[mask, 0].astype(int) - 1)
    surf_dof_idx = filter_good(node_to_dof(surf_nodes))          # ← filtered
    bc_type_id   = int(boundary_list[mask, 5][0])
    bc_name      = bc_type_str.get(bc_type_id, f'type{bc_type_id}')
    if flag in flags_bc_map:
        bc_name = bc_type_str.get(int(flags_bc_map[flag]['bc_type']), bc_name)
    lhs_s = jv_lhs[surf_dof_idx]
    rhs_s = jv_rhs[surf_dof_idx]
    err_s = np.linalg.norm(lhs_s - rhs_s) / (np.linalg.norm(rhs_s) + 1e-14)
    print(f"{flag:<6} {bc_name:<10} {len(surf_nodes):>8} "
          f"{np.linalg.norm(lhs_s):>12.4e} "
          f"{np.linalg.norm(rhs_s):>12.4e} "
          f"{err_s:>12.4e}")

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n{'Region':<30} {'rel_err':>12}")
print("-" * 44)
print(f"{'Global (bad nodes excluded)':<30} {rel_err:>12.4e}")
print(f"{'Interior nodes only':<30} {rel_err_int:>12.4e}")
print(f"{'All boundary nodes':<30} {rel_err_bc:>12.4e}")
for flag in unique_flags:
    mask         = flags == flag
    bc_type_id   = int(boundary_list[mask, 5][0])
    bc_name      = bc_type_str.get(bc_type_id, f'type{bc_type_id}')
    surf_nodes   = np.unique(boundary_list[mask, 0].astype(int) - 1)
    surf_dofs    = filter_good(node_to_dof(surf_nodes))          # ← filtered
    err_s        = (np.linalg.norm(jv_lhs[surf_dofs] - jv_rhs[surf_dofs])
                    / (np.linalg.norm(jv_rhs[surf_dofs]) + 1e-14))
    print(f"  Flag {flag} ({bc_name}){'':<15} {err_s:>10.4e}")

eps for J*q: 1e-08
By central diff, h=2.08e-04
JV relative error (central diff, h=2.08e-04): 1.3628e-01
  ||jv_lhs|| = 2.723039e+13
  ||jv_rhs|| = 2.902063e+13
  Bad nodes excluded: 0

Interior nodes (485241, bad excluded) rel err: 1.3352e-01
  ||jv_lhs|| = 2.4058e+13
  ||jv_rhs|| = 2.5902e+13

Boundary nodes (54915, bad excluded) rel err: 1.4659e-01

Flag   BC type     N nodes      ||lhs||      ||rhs||      rel_err
----------------------------------------------------------------
1      noslip         9919   4.0890e+12   4.3881e+12   3.9856e-01
2      slip          22710   8.6851e+12   8.8654e+12   6.6599e-02
3      slip          22784   8.4445e+12   8.6168e+12   7.1753e-02
4      riemann         472   5.0452e+09   5.0445e+09   3.9009e-04
5      riemann         184   2.8137e+09   2.8134e+09   5.6527e-04

Region                              rel_err
--------------------------------------------
Global (bad nodes excluded)      1.3628e-01
Interior nodes only              1.3352e-01
All bou

In [17]:
# ── Per-node relative JV error ────────────────────────────────────────────────
err_per_node = np.linalg.norm((jv_rhs - jv_lhs).reshape(N, 5), axis=1)
ref_per_node = np.linalg.norm(jv_rhs.reshape(N, 5), axis=1)

rel_err_per_node = err_per_node / (ref_per_node + 1e-20)
log_rel_err      = np.log10(rel_err_per_node + 1e-20)

# ── Attach scalars to mesh ────────────────────────────────────────────────────
block.point_data['jv_rel_error']     = rel_err_per_node
block.point_data['jv_log_rel_error'] = log_rel_err

# ── Interior node overlay ─────────────────────────────────────────────────────
has_interior = interior_node_idx.size != 0
interior_pts = pv.PolyData(block.points[interior_node_idx]) if has_interior else None
interior_set = set(interior_node_idx.tolist()) if has_interior else set()

# ── Plot ──────────────────────────────────────────────────────────────────────
def print_jv_pick(point):
    pid  = block.find_closest_point(point)
    rel  = block.point_data['jv_rel_error'][pid]
    lrel = block.point_data['jv_log_rel_error'][pid]
    xy   = block.points[pid, :2]
    bc   = 'interior' if pid in interior_set else 'boundary'
    print(f"Node {pid:5d}  ({bc})  x={xy[0]:+.4f}  y={xy[1]:+.4f}  "
          f"rel_err={rel:.3e}  log10={lrel:.2f}")

pl = pv.Plotter(notebook=True)
# pl.camera.parallel_projection = True

pl.add_mesh(block,
            scalars='jv_log_rel_error',
            cmap='RdBu_r',
            show_edges=False,
            # clim=[-4, 0],
            edge_color='grey',
            opacity=0.95,
            pickable=True)

if has_interior:
    pl.add_mesh(interior_pts,
                color='yellow',
                point_size=3,
                render_points_as_spheres=True,
                label='Interior nodes')
    pl.add_legend(size=(0.18, 0.07))
else:
    print("No interior nodes present — skipping interior overlay and legend")

pl.enable_point_picking(callback=print_jv_pick,
                        show_message=True,
                        font_size=10,
                        color='black',
                        point_size=10)
pl.view_xy
pl.show()

Widget(value='<iframe src="http://localhost:59857/index.html?ui=P_0x1a9b4d91160_0&reconnect=auto" class="pyvis…